In [17]:
import sys
sys.path.append("../")

In [18]:
import torch
from qmpsqsc.models import mpsqsc
from qmpsqsc.models import qmps
from importlib import reload
from qmpsqsc.models.data.utils import flip_sites_in_mps
import qmpsqsc.models.data as mpsdata

reload(mpsqsc)

<module 'qmpsqsc.models.mpsqsc' from '/Users/keisuke/Documents/projects/mps4qsc/notebooks/../qmpsqsc/models/mpsqsc/__init__.py'>

In [ ]:
import torch.nn.functional as F

L = 30
chi = 2
d = 2
ghz = mpsqsc.build_ghz_state(L, d, chi)
ghz = ghz.normalize()

ghz2 = ghz.copy()
ghz2.As[0][:, 1] = -ghz2.As[0][:, 1]

ghz_errors = [flip_sites_in_mps(ghz, [i]) for i in range(L)]
for ghz_error in ghz_errors:
    ghz_error.normalize()

ghz2_errors = [flip_sites_in_mps(ghz2, [i]) for i in range(L)]
for ghz2_error in ghz2_errors:
    ghz2_error.normalize()

ghzs1 = mpsqsc.add_mpstates([ghz] + ghz_errors)
ghzs2 = mpsqsc.add_mpstates([ghz2] + ghz2_errors)
ghzs1 = ghzs1.normalize()
ghzs2 = ghzs2.normalize()


allup = mpsqsc.build_classical_state(L, d, [0]*L)
alldown = mpsqsc.build_classical_state(L, d, [1]*L)

allup_errors = [flip_sites_in_mps(allup, [i]) for i in range(L)]
alldown_errors = [flip_sites_in_mps(alldown, [i]) for i in range(L)]

mixed_states = mpsqsc.add_mpstates([allup, alldown] + allup_errors + alldown_errors)
mixed_states = mixed_states.normalize()





In [4]:
from qmpsqsc.models.mpsqsc.compress import compress_mpstate

ghzs1, ghzs1_fid = compress_mpstate(ghzs1, 2, adam_steps=0, n_sweeps=1)
ghzs2, ghzs2_fid = compress_mpstate(ghzs2, 2, adam_steps=0, n_sweeps=1)

print(ghzs1_fid, ghzs2_fid)


/Users/keisuke/miniconda3/envs/mpsqsc/lib/python3.11/site-packages/tenpy/networks/mps.py:1629: UserWarning: unit_cell_width is a new argument for MPS and similar classes. It is optional for now, but will become mandatory in a future release. The default value (unit_cell_width=len(sites)) is correct, iff the lattice is a Chain. For other lattices, it is incorrect. It is used for dipolar charges and correlation_function2.
  super().__init__(sites, bc, unit_cell_width)
/Users/keisuke/miniconda3/envs/mpsqsc/lib/python3.11/site-packages/tenpy/algorithms/mps_common.py:2259: UserWarning: VariationalCompression with min_sweeps=max_sweeps: we recommend to set tol_theta_diff=None to avoid overhead
  warnings.warn(


0.7149158415952889 0.7149158415952882


In [5]:
data_generator = mpsdata.ghz.create_ghz_rho_batch_qsc(ghz, allup, alldown, 2**5, 0.5)

In [6]:
ghzs1.set_requires_grad(True)
ghzs2.set_requires_grad(True)
optimizer = torch.optim.Adam(ghzs1.As + ghzs2.As, lr=0.001)

optimizer.zero_grad()

for _ in range(100):
    states, labels, _ = next(data_generator)
    loss, acc = mpsdata.calculate_loss_mpstates(ghzs1, ghzs2, states, labels)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    print(loss.item(), acc.item())


0.34657359027997314 0.6875
0.29091450685712195 1.0
0.24337621903878115 1.0
0.20203794001485376 1.0
0.16789767580853404 1.0
0.13784147876193142 1.0
0.1153650594015408 1.0
0.09491307688849739 1.0
0.08086398299926278 1.0
0.06638839437377017 1.0
0.05616587367500496 1.0
0.04721652506098546 1.0
0.039857199185197936 1.0
0.03428494995963128 1.0
0.02937603832617473 1.0
0.025666786167257247 1.0
0.022491947585659002 1.0
0.019381590210702238 1.0
0.017569177972361218 1.0
0.01624066060672292 1.0
0.014422848768636703 1.0
0.012900826655631864 1.0
0.011888897218020247 1.0
0.011150817975242332 1.0
0.009809322280311273 1.0
0.00946327254114844 1.0
0.008700006942121646 1.0
0.008016436059845027 1.0
0.007920636983996296 1.0
0.007027701892264404 1.0
0.006735133269905299 1.0
0.006508781381117378 1.0
0.005890106910443893 1.0
0.0058179503076171216 1.0
0.005586746556868701 1.0
0.005425635745919007 1.0
0.005231357842154314 1.0
0.004813484998828671 1.0
0.005383464805197349 1.0
0.0060271365553893 1.0
0.0052435185738

In [7]:
ghz_qsc = mpsqsc.helper.build_qsc_from_mpstates(ghzs1, ghzs2)

In [8]:
ghz_qsc.set_requires_grad(True)

In [9]:
ghz_qsc_chi = ghz_qsc.truncate_bond_dimension(2)

In [10]:
ghz_qsc_chi.set_requires_grad(True)

In [11]:
optimizer = torch.optim.Adam(ghz_qsc_chi.As, lr=0.001)

optimizer.zero_grad()

for _ in range(100):
    states, labels, _ = next(data_generator)
    loss, acc = mpsdata.calculate_loss_mpsqsc(ghz_qsc_chi, states, labels)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    print(loss.item(), acc.item())


0.006922370534914022 1.0
0.07782870532483009 1.0
0.009038360665340356 1.0
0.030182052188885748 1.0
0.049784888744396766 1.0
0.030232731777893002 1.0
0.01013097743836257 1.0
0.007924735441950945 1.0
0.019729564229326126 1.0
0.025837136393066087 1.0
0.020254780624242866 1.0
0.01716048332758382 1.0
0.015435735278966144 1.0
0.014034080772436494 1.0
0.013394157841859627 1.0
0.011465015648310192 1.0
0.00782469092596799 1.0
0.008978158211786793 1.0
0.00586842218750449 1.0
0.010600388806210408 1.0
0.00909907767429867 1.0
0.010588791954388624 1.0
0.009246769588739639 1.0
0.022252414733593132 1.0
0.006652391827972346 1.0
0.011539746610549094 1.0
0.010905435004972413 1.0
0.017239054425004407 1.0
0.008326573611008803 1.0
0.005078331887080014 1.0
0.007478841991945804 1.0
0.009508834197554591 1.0
0.009316559125742363 1.0
0.007297076593272609 1.0
0.00845792180890857 1.0
0.004739192487169528 1.0
0.006026722869144094 1.0
0.0065639710645421755 1.0
0.006206766422830632 1.0
0.006503089290748911 1.0
0.0054

# Save MPS qsc

In [16]:
ghz_qsc_chi.save_to("data/ghz_qsc_error.pt")